In [32]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [33]:
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os
from dotenv import load_dotenv
import copy
load_dotenv()

# Load a sample dataset
from datasets import load_dataset

True

In [34]:
login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [35]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

Using device: mps


In [36]:
# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [37]:
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

In [38]:
tokenizer

GPT2TokenizerFast(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|im_start|>', 'eos_token': '<|im_end|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|im_end|>', 'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=

Generate with the base model
Here we will try out the base model which does not have a chat template.

In [39]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)
print(f"Formatted prompt: {formatted_prompt}")

Formatted prompt: <|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>



In [40]:
# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=30)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

Before training:
<|im_start|>user
Salt and sugar can look very similar, but they taste very different. Can you tell me how to distinguish between them?<|im_end|>

The salt and sugar are both salts, but they are not the same. Salt is a mineral, while sugar is a carbohydrate. Salt is a


In [41]:
# Use a reasoning dataset
ds = load_dataset("prithivMLmods/Deepthink-Reasoning")

ds

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response'],
        num_rows: 251
    })
})

In [42]:
ds["train"].shuffle().select([1])[:1]

{'prompt': ['Write a C# program that implements a basic calculator, with functions for addition, subtraction, multiplication, and division. The calculator should accept inputs for two numbers and return the result of the chosen operation. Use object-oriented programming principles to design the program.'],
 'response': ['<|thinking|>\n## Analyzing the request\nThe user wants a C# program that implements a basic calculator with addition, subtraction, multiplication, and division. The program should use object-oriented programming (OOP) principles.\n\n## Planning the solution\n1. Define a `Calculator` class to encapsulate the calculator\'s functionality.\n2. Implement methods for addition, subtraction, multiplication, and division.\n3. Use a `Main` method to handle user input and display the result.\n\n## Deciding on the approach\nI will write a C# program that:\n1. Defines a `Calculator` class with methods for the four operations.\n2. Uses a `Main` method to interact with the user and c

In [43]:
def tokenize_function(examples):
    prompts = [p.strip() for p in examples["prompt"]]
    responses = [r.strip() for r in examples["response"]]
    texts = [
        tokenizer.apply_chat_template(
            [{"role": "user", "content": p}, {"role": "assistant", "content": r}],
            tokenize=False
        )
        for p, r in zip(prompts, responses)
    ]
    return tokenizer(texts, truncation=True, padding="max_length", max_length=512, )

ds = ds.map(
    tokenize_function,
    batched=True,
    desc="Tokenizing dataset",
)

In [44]:
print(ds["train"][2]["prompt"])

print("=====" * 100)

print(ds["train"][2]["response"])


Create a Python script to print the numbers from 1 to 50, but for multiples of 3 print "Fizz" instead of the number, for multiples of 5 print "Buzz" instead of the number and for multiples of both 3 and 5 print "FizzBuzz".

Not applicable
<|thinking|>
## Thinking about the FizzBuzz problem

The problem is a classic coding challenge.

## Key Steps

1. **Loop through numbers 1 to 50:** A `for` loop will be used to iterate through the numbers.
2. **Check for multiples of 3 and 5:** The modulo operator (`%`) will be used to check if a number is divisible by 3 or 5.
3. **Conditional Printing:** `if`, `elif`, and `else` statements will be used to print "Fizz", "Buzz", "FizzBuzz", or the number itself based on the conditions.

## Python Code

```python
def fizzbuzz():
  """Prints numbers from 1 to 50, replacing multiples of 3 with "Fizz",
  multiples of 5 with "Buzz", and multiples of both with "FizzBuzz".
  """
  for i in range(1, 51):
    if i % 3 == 0 and i % 5 == 0:
      print("FizzBuzz"

In [54]:
finetune_name = "SmolLM2-FT-MyDataset"

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output",
    max_steps=400,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=20,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    # eval_strategy="steps",  # Evaluate the model at regular intervals
    # eval_steps=50,  # Frequency of evaluation
    use_mps_device=(
        True if device == "mps" else False
    ),  # Use MPS for mixed precision training
    hub_model_id=finetune_name,  # Set a unique name for your model
   
)

# create a copy of the model to avoid modifying the original
copy_model = copy.deepcopy(model)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=copy_model,
    args=sft_config,
    train_dataset=ds["train"],
   #  eval_dataset=ds["train"],
)


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/transformers/training_args.py:2214: UserWarning: `use_mps_device` is deprecated and will be removed in version 5.0 of 🤗 Transformers. `mps` device will be used by default if available similar to the way `cuda` device is used.Therefore, no action from user is required. 
  warnings.warn(


In [55]:
trainer.train()

/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
20,1.440800
40,1.255000
60,1.014700
80,0.920700
100,0.864400
120,0.801200
140,0.721800
160,0.621400
180,0.720900
200,0.657500


/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/siddhartha.banerjee/Documents/personal/tiny-tune/tiny-tune_env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=400, training_loss=0.6962287330627441, metrics={'train_runtime': 348.3788, 'train_samples_per_second': 4.593, 'train_steps_per_second': 1.148, 'total_flos': 520053684830208.0, 'train_loss': 0.6962287330627441})

In [58]:
# Let's test the base model before training
prompt = "Salt and sugar can look very similar, but they taste very different. \
          Can you tell me how to distinguish between them?"


formatted_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], tokenize=False
)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

# Create a streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)


outputs = model.generate(**inputs, max_new_tokens=100, streamer=streamer)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("\n\n ====== Training complete, generating new outputs ======")

outputs = trainer.model.generate(**inputs, max_new_tokens=100, streamer=streamer)
print("After training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("done")


A. Yes, salt and sugar are both salts. Salt is a chemical compound that is made up of sodium and chlorine. Sugar is a simple sugar that is made up of glucose and fructose.

B. Salt is a salt. Salt is a chemical compound that is made up of sodium and chlorine. Sugar is a simple sugar that is made up of glucose and fructose.

C. Salt is a salt. Salt is a chemical compound that is made up of sodium and chlorine.
Before training:
user
Salt and sugar can look very similar, but they taste very different.           Can you tell me how to distinguish between them?

A. Yes, salt and sugar are both salts. Salt is a chemical compound that is made up of sodium and chlorine. Sugar is a simple sugar that is made up of glucose and fructose.

B. Salt is a salt. Salt is a chemical compound that is made up of sodium and chlorine. Sugar is a simple sugar that is made up of glucose and fructose.

C. Salt is a salt. Salt is a chemical compound that is made up of sodium and chlorine.


 ====== Training com